# Notebook 8: annotations and street names

Phase 4, second part. Notebook 7/7b worked out floors, walls and a window grid, and how to describe a spot both ways ("floor 19, south wall, leftmost" ↔ a 3D point). This notebook uses that to **save your own notes on spots** ("my flat"), adds **street names** to the 3D view, and asks two questions about an annotated window: **what blocks its view**, and **how many hours of sun it gets**.

| Step | What we do |
|---|---|
| **5** | The annotation file: what's in it and why |
| **6** | Add, show, edit and delete annotations, also by clicking a wall |
| **7** | Street names: find the data, check the coordinates, place 3D labels |
| **8** | Less clutter: show/hide controls, and how the labels affect smoothness |
| **9** | View check (what blocks the view from a window) and sun exposure (hours of direct sun) |

Everything from Notebook 7/7b comes from `citymap/` (`floors.py`, `walls.py`, `windows.py`); new here: **`citymap/annotations.py`**, **`citymap/streets.py`**, and for Step 9 **`citymap/rays.py`** and **`citymap/sun.py`**.

> **Estimates:** floor levels and window columns are approximate (the models have no floors or windows); estimated floor counts have a `~`. An annotation is saved as a *description* (floor, wall, column), so its 3D position is always "the approximate spot of that description".

Run from top to bottom after a kernel restart. **One live 3D view at a time.**

**Before the first run:** Step 7 needs one new library, `pyproj` (explained in 7.1). Install it once, in a terminal in the project folder: `.venv/bin/python -m pip install -r requirements.txt`

### Glossary (new words in this notebook)

| Word | Meaning |
|---|---|
| **Annotation** | Your own label on a spot of a building, with a note and a color, e.g. "my flat". |
| **JSON** | A plain-text file format for lists and dictionaries (`{"label": "my flat"}`). Readable and editable in any text editor. |
| **Marker** | The colored ball drawn on the wall at an annotation's spot. |
| **Street segment** | One piece of a street's center line, usually one block, between two crossings. |
| **Centerline** | The line along the middle of a street. |
| **CFCC** | A US Census code for the road class: A1x = highway, A2x/A3x = major road, A4x = local street. |
| **Coordinate system (EPSG code)** | The rule that turns positions into numbers. EPSG:2249 = Massachusetts State Plane, US feet (our 3D data); EPSG:4326 = latitude/longitude. |
| **pyproj** | A Python library that converts positions between coordinate systems. |
| **Toggle** | A control that switches something on or off (a checkbox). |
| **Clutter** | So many labels that they cover each other and the city. |
| **Frame time** | How long it takes to draw the 3D view once. Under ~50 ms feels smooth when rotating. |
| **Ray** | A straight line from a point in one direction. "Which building does the ray meet first?" is the question behind both parts of Step 9. |
| **Fan of rays** | Many rays from one point, a few degrees apart, like the ribs of a hand fan. |
| **Data area** | The rectangle covered by our 12 tiles. Beyond it the model has no buildings, so we don't know what's there. |
| **Azimuth** | The sun's compass bearing: 0° = north, 90° = east, 180° = south. |
| **Elevation (of the sun)** | How high the sun stands above the horizon: 0° = on the horizon, 90° = straight up. |
| **Solar noon** | When the sun is highest, due south. In Boston about 11:45 in winter and 12:45 in summer (daylight saving time). |
| **Solstice / equinox** | Dec 21: lowest sun, shortest day. Jun 21: highest sun, longest day. Mar 20 (and Sep 22): day and night equally long. |

## Setup *(boilerplate)*

As in Notebook 7b, plus the new modules (`annotations`, `streets`, and for Step 9 `rays` and `sun`), `ipywidgets` (the form and controls under the view, as in Notebooks 5 and 6) and `shapely` (for the street alignment check in 7.4). `date` is for the sun dates in Step 9.

In [1]:
%load_ext autoreload
%autoreload 2

import html
import json
import sys
import time
from datetime import date
from pathlib import Path

import ipywidgets
import numpy as np
import pandas as pd
import pyvista as pv
import shapely
from IPython.display import display

sys.path.insert(0, str(Path("..").resolve()))   # so Python finds citymap/ in the project folder
from citymap import annotations, config, floors, footprints, loading, lookup, rays, streets, sun, viewing, windows
from citymap import walls as wall_tools         # ("walls" is a handy name for wall tables)

pv.set_jupyter_backend("server")   # picture-only viewer: fast updates (see Notebook 2, Step 5)

## Load everything *(as in Notebook 7b)*

**What this does:** the catalog, the 12 tiles with the aerial photo, the lookup `Model_ID` → (tile, index), and the floor data.

**What to look for:** 21,369 catalog rows, 12 tiles, 19,591 buildings, 19,591 floor rows, in a few seconds.

In [2]:
start = time.perf_counter()
tile_names = [tile for tile in config.BLOCK_TILES if loading.tile_is_downloaded(tile)]
catalog = lookup.load_catalog(tile_names)

tiles = {}
for tile in tile_names:
    tiles[tile] = loading.load_tile(tile, ground="aerial", merged=True)   # from data/cache/
locations = lookup.building_locations(tiles)
floor_data = floors.load_floor_data()                 # floor counts + your overrides (Notebook 7)

print(f"Catalog: {len(catalog):,} rows.  3D: {len(tiles)} tiles, {len(locations):,} buildings."
      f"  Floors: {len(floor_data['counts']):,} rows.  ({time.perf_counter() - start:.1f} s)")

Catalog: 21,369 rows.  3D: 12 tiles, 19,591 buildings.  Floors: 19,591 rows.  (3.1 s)


## Style and test buildings *(as in Notebook 7b)*

**Output:** nothing.

In [3]:
STYLE = {
    "background": "#eceded", "flat_ground": True, "building_color": "#fbfaf6",
    "building_ambient": 0.35, "building_diffuse": 0.75, "outline_color": "#8a8a86", "outline_opacity": 0.6,
}
TEST_BUILDINGS = {
    "100 Federal": "BOS-O0J6Z-FLDEJ-5OM89",
    "Hancock": "BOS-P65JM-OO4A1-HYDGP",
    "One Dalton": "BOS-P7Y4F-KOTUV-BCJSU",
}


def building_name(model_id):
    row = lookup.get_building(catalog, model_id)
    return "(no name)" if row is None or pd.isna(row["Name"]) else row["Name"]

---
## Step 5: the annotation file

Annotations are saved in **`data/annotations/annotations.json`**: one plain-text file, a list with one entry per annotation.

**Kept out of git** (your decision): it may contain personal places. Nothing had to change: `.gitignore` already ignores everything in `data/*/`, which also keeps `floor_overrides.csv` local.

### Why the description, not the 3D point

Each annotation stores **where it is in words** (building, floor, wall direction, column), not x/y/z. The 3D point is worked out again every time the notebook runs (`windows.locate`, Notebook 7b, Step 4). So if the floor data or the wall code improves, annotations move with it and stay on "floor 19, south wall", instead of floating at an old position.

```
  annotations.json            every run                         3D view
  ┌──────────────────┐   windows.locate(model, floor,     ┌────────────────┐
  │ model_id, floor, │ ──  direction, column, wall_no) ──► │ ● my flat      │
  │ direction, column│                                     │ (on the wall)  │
  └──────────────────┘                                     └────────────────┘
```

### The fields

| Field | Example | Meaning |
|---|---|---|
| `id` | `1` | a number, given when it's added; used to edit or delete it |
| `model_id` | `"BOS-P65JM-OO4A1-HYDGP"` | the building (the catalog's `Model_ID`) |
| `floor` | `19` | floor number (1 = ground floor). Its height is estimated |
| `direction` | `"S"` | which way the wall faces: N, NE, E, SE, S, SW, W, NW |
| `column` | `1` or `"leftmost"` | which ~10 ft strip of the wall, from the left as seen from outside (the description then says it as a distance); or `"leftmost"`, `"rightmost"`, `"center"` |
| `wall_number` | `1` | *(added to the plan's list)* which of the walls facing that way: 1 = the widest. Without it, a spot clicked on a second, narrower south wall would jump to the widest one on the next run |
| `label` | `"my flat"` | the short text shown in the 3D view |
| `note` | `"2 windows, corner"` | a longer note, shown when you click the marker |
| `color` | `"#e0218a"` | the marker's color (a hex code) |
| `created` | `"2026-11-06T14:03:00"` | when it was added (date and time) |

**What this does:** shows one example annotation as it would look in the file (not saved).

**What to look for:** the fields above, in JSON.

In [4]:
example = {"id": 1, "model_id": TEST_BUILDINGS["Hancock"], "floor": 19, "direction": "S", "column": "leftmost",
           "wall_number": 1, "label": "my flat", "note": "2 windows, south end", "color": "#e0218a",
           "created": "2026-11-06T14:03:00"}
print(json.dumps([example], indent=2))     # the file is a list of these

[
  {
    "id": 1,
    "model_id": "BOS-P65JM-OO4A1-HYDGP",
    "floor": 19,
    "direction": "S",
    "column": "leftmost",
    "wall_number": 1,
    "label": "my flat",
    "note": "2 windows, south end",
    "color": "#e0218a",
    "created": "2026-11-06T14:03:00"
  }
]


**What this does:** where the file is, and what's in it now (`annotations.list_annotations()`: a table, one row per annotation).

**What to look for:** on the very first run, `exists: False` and an empty table. After that, your annotations.

In [5]:
print(annotations.ANNOTATIONS_FILE, " exists:", annotations.ANNOTATIONS_FILE.exists())
annotations.list_annotations()

/Users/lilit/.claude/city-mapping/data/annotations/annotations.json  exists: True


,id,model_id,floor,direction,column,wall_number,label,note,color,created
0,1,BOS-P65JM-OO4A1-HYDGP,19,S,leftmost,1,my flat (example),"The plan's example (Notebook 8, Step 6), note ...",#e0218a,2026-11-06T17:52:31


---
## Step 6: add, show, edit and delete annotations

The functions, in `citymap/annotations.py`. Each one **reads the file, changes it and saves it right away**, so nothing is lost if the kernel stops:

| Function | What |
|---|---|
| `add_annotation(model_id, floor, direction, column, label, note="", color=..., wall_number=1)` | adds one; returns it with its new `id` |
| `list_annotations()` | all of them, as a table |
| `update_annotation(id, note="...", label="...", ...)` | changes the fields you name (not `id` or `created`) |
| `delete_annotation(id)` | removes one |
| `place_annotations(tiles, locations, floor_data, list)` | works out each one's 3D point (`windows.locate`) |

### 6.1 The example: "my flat"

**What this does:** adds the plan's example, *my flat (example)* on the Hancock tower, floor 19, south wall, leftmost column, **only if it isn't in the file yet** (so running the notebook again doesn't add it twice).

**What to look for:** the table now has a row *my flat (example)*. On later runs: the same row, with its original `created` time: proof that it was saved.

In [6]:
EXAMPLE_LABEL = "my flat (example)"

if EXAMPLE_LABEL not in set(annotations.list_annotations()["label"]):      # only the first time
    annotations.add_annotation(TEST_BUILDINGS["Hancock"], 19, "S", "leftmost", EXAMPLE_LABEL,
                               note="The plan's example (Notebook 8, Step 6). Delete it whenever you like.")
annotations.list_annotations()

,id,model_id,floor,direction,column,wall_number,label,note,color,created
0,1,BOS-P65JM-OO4A1-HYDGP,19,S,leftmost,1,my flat (example),"The plan's example (Notebook 8, Step 6), note ...",#e0218a,2026-11-06T17:52:31


### 6.2 Edit and delete

**What this does:**
- changes the example's note with `update_annotation`;
- adds a temporary annotation (100 Federal, floor 5, west wall, column 3) and deletes it again with `delete_annotation`.

**What to look for:** "added id …" and "deleted id …" with the same number, and the table as before, with the example's new note.

In [7]:
table = annotations.list_annotations()
example_id = int(table[table["label"] == EXAMPLE_LABEL]["id"].iloc[0])
annotations.update_annotation(example_id, note="The plan's example (Notebook 8, Step 6), note changed with update_annotation.")

temporary = annotations.add_annotation(TEST_BUILDINGS["100 Federal"], 5, "W", 3, "temporary")
print("added id", temporary["id"])
print("deleted id", annotations.delete_annotation(temporary["id"])["id"])
annotations.list_annotations()

added id 2
deleted id 2


,id,model_id,floor,direction,column,wall_number,label,note,color,created
0,1,BOS-P65JM-OO4A1-HYDGP,19,S,leftmost,1,my flat (example),"The plan's example (Notebook 8, Step 6), note ...",#e0218a,2026-11-06T17:52:31


### 6.3 From description to 3D point

**What this does:** `place_annotations` runs `windows.locate` for every annotation in the file. Those that can't be placed (a building outside the loaded tiles, a floor that no longer exists after you changed the floor data) are listed with the reason, not dropped silently.

**What to look for:** *my flat (example)* → *Floor ~19 of 60 (manual count, level estimated), south-facing wall (159°), about 5 ft from the wall's left end (wall ~60 ft wide)*, and its 3D point, about (−12525, −7932, 253). No problems.

In [8]:
placed, problems = annotations.place_annotations(tiles, locations, floor_data, annotations.load_annotations())
for annotation, spot in placed:
    print(f"{annotation['id']:3}  {annotation['label']:20} {spot['description']}\n{'':25} point {np.round(spot['point'], 1)}")
for annotation, reason in problems:
    print(f"{annotation['id']:3}  {annotation['label']:20} NOT PLACED: {reason}")

  1  my flat (example)    Floor ~19 of 60 (manual count, level estimated), south-facing wall (159°), column 1 of 6 from the left
                          point [-12525.4  -7931.5    253.2]


### 6.4 What the map shows: `DISPLAY` and `refresh_overlays`

One settings dict says what's drawn on top of the city. Street names come in Step 7 and the controls in Step 8; they're all here so there's a single place to look.

| Setting | Meaning |
|---|---|
| `annotations` | show the markers and labels |
| `street_names` | show street names (Step 7) |
| `street_classes` | only these road classes, e.g. `["highway", "major"]`; `None` = all |
| `min_street_length_ft` | only streets at least this long (one connected piece) |
| `tallest_buildings` | name this many of the tallest buildings (0 = none) |

**What this does:**
- `tallest_building_labels(count)`: names and label points (30 ft above the roof) of the tallest buildings;
- `refresh_overlays()`: places the annotations again (`place_annotations`, so new ones appear) and redraws everything `DISPLAY` asks for. Each overlay has a **name** in the view, so redrawing replaces it instead of adding a copy.

**Output:** nothing.

In [9]:
DISPLAY = {
    "annotations": True,
    "street_names": True,
    "street_classes": ["highway", "major"],   # None = all classes
    "min_street_length_ft": 300,
    "tallest_buildings": 0,
}
street_labels = None          # all street label positions: made in Step 7


def tallest_building_labels(count):
    loaded = catalog[(catalog["Status"] == "Current") & catalog["Model_ID"].isin(list(locations))]
    tallest = loaded.drop_duplicates("Model_ID").nlargest(count, "Height_Ft")
    points, names = [], []
    for index, row in tallest.iterrows():
        tile, building = locations[row["Model_ID"]]
        mesh = tiles[tile]["buildings"]
        x_min, x_max, y_min, y_max, z_min, z_max = mesh.extract_cells(mesh.cell_data["building_index"] == building).bounds
        points.append([(x_min + x_max) / 2, (y_min + y_max) / 2, z_max + 30])   # above the roof
        names.append(building_name(row["Model_ID"]))
    return np.array(points), names

In [10]:
def refresh_overlays():
    global placed, shown_street_labels
    placed, problems = annotations.place_annotations(tiles, locations, floor_data, annotations.load_annotations())
    annotations.add_annotation_markers(map_plotter, placed if DISPLAY["annotations"] else [])

    shown_street_labels = []
    if street_labels is not None and DISPLAY["street_names"]:
        shown_street_labels = streets.choose_labels(street_labels, DISPLAY["street_classes"], DISPLAY["min_street_length_ft"])
    streets.add_street_labels(map_plotter, pd.DataFrame(shown_street_labels))   # empty: removes them

    map_plotter.remove_actor("tallest_names")
    if DISPLAY["tallest_buildings"] > 0:
        points, names = tallest_building_labels(DISPLAY["tallest_buildings"])
        map_plotter.add_point_labels(points, names, font_size=12, show_points=False, shape_color="#e8eefc",
                                     always_visible=True, name="tallest_names")
    map_plotter.render()

### 6.5 Add an annotation by clicking a wall

**How it works:**

```
 click ──► near a marker (within 15 ft)?  ── yes ──► show that annotation: label, spot, NOTE
                    │ no
                    ▼
           windows.describe_point (Notebook 7b, Step 4)
                    │
        on a wall? ── no ──► say why (roof, minor wall)
                    │ yes
                    ▼
   the spot in words + yellow cell  ──►  you check it, type a label (and a note), press "Save annotation"
                                                                    │
                                           add_annotation(...) ──► saved, marker appears
```

Markers sit 2 ft in front of the wall and can't be hit by a click themselves, so "clicking a marker" = clicking the wall right behind it.

**What this does:** the form under the view (boilerplate) and the two functions that run on a click and on "Save annotation".

**Output:** nothing.

In [11]:
# The form under the view (boilerplate: what each control looks like)
info_box = ipywidgets.HTML("Click a wall to describe it, or a marker to read its note.")
label_text = ipywidgets.Text(placeholder="label, e.g. my flat", description="Label:")
note_text = ipywidgets.Text(placeholder="note (optional)", description="Note:", layout=ipywidgets.Layout(width="360px"))
color_picker = ipywidgets.ColorPicker(value=annotations.DEFAULT_COLOR, description="Color:", concise=True)
save_button = ipywidgets.Button(description="Save annotation", icon="save", disabled=True)
form_message = ipywidgets.HTML("")
clicked_spot = None           # (model_id, spot) of the last wall click, or None

In [ ]:
def show_lines(lines):
    viewing.show_text_panel(map_plotter, lines)                                 # in the view
    info_box.value = "<br>".join(html.escape(line) for line in lines)           # under the view


def on_map_click(tile, model_id, point):
    global clicked_spot
    clicked_spot = None
    map_plotter.remove_actor("chosen_cell")
    map_plotter.remove_actor("chosen_cell_label")
    near = annotations.nearest_annotation(placed, point) if model_id and DISPLAY["annotations"] else None
    if model_id is None:
        lines = ["Click a wall to describe it, or a marker to read its note."]
    elif near is not None:                                                    # a marker: its note
        annotation, spot = near
        lines = [f"{annotation['label']}  (annotation {annotation['id']})", spot["description"],
                 f"Note: {annotation['note'] or '(none)'}", f"Added {annotation['created']}"]
    else:                                                                     # a wall: describe it
        spot = windows.describe_point(tiles, locations, floor_data, model_id, point)
        lines = [building_name(model_id), spot["description"]]
        if spot["point"] is not None:
            clicked_spot = (model_id, spot)
            windows.add_chosen_cell(map_plotter, spot["frame"], spot["cell"], "selected: type a label below to save")
            lines.append("To save it: type a label below and press 'Save annotation'.")
    save_button.disabled = clicked_spot is None
    show_lines(lines)
    map_plotter.render()

In [13]:
def on_save(button):
    if clicked_spot is None:
        return
    if not label_text.value.strip():
        form_message.value = "<b>Type a label first.</b>"
        return
    model_id, spot = clicked_spot
    new = annotations.add_annotation(model_id, spot["floor"], spot["direction"], spot["column"],     # core: saved
                                     label_text.value.strip(), note_text.value.strip(), color_picker.value,
                                     wall_number=spot["wall_number"])
    form_message.value = html.escape(f"Saved annotation {new['id']}: {new['label']} ({spot['description']})")
    label_text.value, note_text.value = "", ""
    on_map_click(None, None, None)                    # clear the yellow cell and the form
    refresh_overlays()                                # the new marker appears


save_button.on_click(on_save)

### 6.6 The map view

**What this does:** `show_map(view)`: a new view with the city and the aerial photo, everything `DISPLAY` asks for, clicks switched on (6.5), and the form under it. The first view looks at the example annotation, from in front of its wall (`windows.wall_view`, Notebook 7b, Step 3.7). Step 9 uses the optional `extra`: a function that draws more (rays, colored cells) before the view is shown.

**What to look for:**
- a **magenta ball** on the Hancock tower's south end, about a third of the way up, at its left (west) edge, with the label *my flat (example)*;
- **click the ball** (or the wall right behind it): the panel shows its spot and **note**;
- **click a wall** somewhere else: its description and a yellow cell. Type a label, press **Save annotation**: a new ball appears there, and the table in 6.1 will have it the next time you run that cell.

**The plan's "done when":** add an annotation here, then **restart the kernel and run all cells**: the ball is still in the same spot.

In [14]:
def show_map(view, extra=None):
    """A new map view. extra: optional function(plotter) that draws more on it before it's shown (Step 9)."""
    global map_plotter
    pv.close_all()                                    # only one live 3D view at a time
    map_plotter = pv.Plotter()
    map_plotter.set_background(STYLE["background"])
    building_actors = viewing.add_tiles(map_plotter, tiles, STYLE["building_color"], flat_ground=STYLE["flat_ground"],
                                        building_ambient=STYLE["building_ambient"], building_diffuse=STYLE["building_diffuse"])
    viewing.add_outlines(map_plotter, tiles, STYLE["outline_color"], STYLE["outline_opacity"])
    refresh_overlays()                                # annotations, street names, tallest names
    if extra is not None:
        extra(map_plotter)                            # e.g. the view check's rays (Step 9)
    viewing.enable_building_click(map_plotter, building_actors, tiles, on_map_click)   # core: clicks -> 6.5
    viewing.go_to_view(map_plotter, view)
    map_plotter.show()
    display(info_box, ipywidgets.HBox([label_text, note_text, color_picker, save_button]), form_message)

In [37]:
example_spot = windows.locate(tiles, locations, floor_data, TEST_BUILDINGS["Hancock"], 19, "S", "leftmost")
show_map(windows.wall_view(example_spot["frame"], example_spot["cell"], distance=600))

Widget(value='<iframe src="http://localhost:50950/index.html?ui=P_0x174adca90_0&reconnect=auto" class="pyvista…

HTML(value='(no name)<br>Floor ~24 of 33 (assessment count, level estimated), south-facing wall (161°), column…

HTML(value='')

---
## Step 7: street names as 3D labels

### The data (reported and approved before downloading)

| | |
|---|---|
| **Dataset** | *Boston Street Segments (SAM System)*, City of Boston (the street centerlines of the city's address system, SAM) |
| **Link** | https://data.boston.gov/dataset/boston-street-segments-sam-system |
| **Downloaded from** | its ArcGIS service, `gisportal.boston.gov/.../SAM/Live_SAM_Address/FeatureServer/3`, which can cut out an area |
| **Format** | GeoJSON: one line per street segment (usually one block) |
| **Coordinate system** | MA State Plane Mainland, **US feet (EPSG:2249)**: the same as the 3D data, so no reprojection |
| **Name columns** | `PRE_DIR` ("W"), `ST_NAME` ("BROADWAY"), `ST_TYPE` ("ST"), `SUF_DIR` |
| **Road class** | yes: `CFCC` (A1x highway, A2x/A3x major road, A4x local street, others = ramps, paths, …) |
| **License** | Open Data Commons **Public Domain** Dedication and License (PDDL): free to use, no conditions |
| **Area** | the 12 tiles plus 500 ft on every side: about 6,000 segments |

OpenStreetMap was the fallback; not needed.

### From the file to our map

```
  State Plane feet          − (731,100, 2,902,900)      − (52,500, 57,500)
  (EPSG:2249, the     ───►  Metro Boston 3D origin ───►  our global origin ───►  3D view coordinates
   street file)             = the 3D files' coordinates
```

`streets.to_view(...)` does both subtractions.

### 7.1 Check the conversion first, with the catalog

The plan: before trusting the street positions, check the conversion on something we know. Every catalog row has its center as **latitude/longitude** (`Centr_Lat`, `Centr_Lon`) and in the **3D files' coordinates** (`Centr_X_Ft`, `Centr_Y_Ft`). If latitude/longitude → State Plane → minus the Metro Boston origin lands on `Centr_X_Ft`/`Centr_Y_Ft`, our understanding of the coordinate systems is right.

**New library: `pyproj`.** It converts positions between coordinate systems; it's the standard tool for this (it uses the PROJ library that most mapping software uses). We tell it "from EPSG:4326 (latitude/longitude) to EPSG:2249 (MA State Plane, US feet)" and it does the map projection math. Added to `requirements.txt`. Only this check needs it: the street file is already in State Plane.

**What this does:** converts 300 random buildings' latitude/longitude and compares with their catalog coordinates.

**What to look for:** median **3.2 ft**, 95% under 3.2 ft, and an average shift of about **+3.1 ft north**, −0.1 ft east (one odd building is about 37 ft off). So nearly every building is off by the **same** 3 ft in the same direction: that's the small difference between two versions of the latitude/longitude reference (the catalog's and pyproj's default), not an error in our shifts. Phase 2 found the same "within about 3 ft". It doesn't affect the streets: they come in State Plane directly, without latitude/longitude. Hundreds of feet would have meant a wrong system or a missed origin.

In [16]:
import pyproj        # coordinate conversions (new in Step 7)

to_state_plane = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:2249", always_xy=True)  # (lon, lat) -> (x, y) ft
sample = catalog[catalog["Status"] == "Current"].sample(300, random_state=1)
x, y = to_state_plane.transform(sample["Centr_Lon"].to_numpy(), sample["Centr_Lat"].to_numpy())
dx = x - loading.STATE_PLANE_SHIFT[0] - sample["Centr_X_Ft"].to_numpy()     # minus the Metro Boston origin
dy = y - loading.STATE_PLANE_SHIFT[1] - sample["Centr_Y_Ft"].to_numpy()
distance = np.hypot(dx, dy)
print(f"Difference: median {np.median(distance):.1f} ft, 95% under {np.percentile(distance, 95):.1f} ft, "
      f"largest {distance.max():.1f} ft.  Average shift: east {dx.mean():+.1f} ft, north {dy.mean():+.1f} ft")

Difference: median 3.2 ft, 95% under 3.2 ft, largest 37.0 ft.  Average shift: east -0.1 ft, north +3.1 ft


### 7.2 Download (only the first time)

**What this does:** `streets.download_streets(tile_names)` asks the city's service for every segment in the rectangle around our 12 tiles plus 500 ft, in pages of 1,000 (it retries if the connection breaks), and saves them as **`data/streets/street_segments.geojson`**, with the source and license written into the file. If the file is already there, nothing is downloaded.

**What to look for:** the first time, after about 30–60 s: *Downloaded 6,060 segments* (the number may change a little as the city updates its data), about 1–2 MB. Later: *Already downloaded*.

In [17]:
if streets.STREETS_FILE.exists():
    print("Already downloaded:", streets.STREETS_FILE)
else:
    start = time.perf_counter()
    count = streets.download_streets(tile_names, margin_ft=500)
    print(f"Downloaded {count:,} segments in {time.perf_counter() - start:.0f} s")
print(f"{streets.STREETS_FILE.stat().st_size / 1e6:.1f} MB")

Already downloaded: /Users/lilit/.claude/city-mapping/data/streets/street_segments.geojson
1.8 MB


### 7.3 Read and convert

**What this does:** `streets.load_streets()` reads the file into a table, one row per segment:
- `name`: the parts put together and tidied, e.g. `W` + `BROADWAY` → *W Broadway*, `FEDERAL` + `ST` → *Federal St*. Segments without a name (some ramps and paths) get `None`;
- `road_class`: from CFCC: *highway*, *major*, *local* or *other*;
- `line`: the segment as a shapely line, **already in 3D view coordinates** (`to_view`).

**What to look for:** about 6,060 segments: ~3,000 local, ~2,200 major, ~700 other, ~120 highway; ~220 without a name. Coordinates around −20,000 to 0 (like the 3D buildings), not 770,000.

In [18]:
segments = streets.load_streets()
print(f"{len(segments):,} segments, {segments['name'].isna().sum()} without a name")
print(segments["road_class"].value_counts().to_dict())
segments.head()

6,060 segments, 223 without a name
{'local': 3002, 'major': 2223, 'other': 718, 'highway': 117}


,segment_id,name,road_class,cfcc,line
0,1,A St,major,A31,"LINESTRING (-7594.599999999977 -10882, -7484.4..."
1,2,A St,major,A31,"LINESTRING (-7484.400000000023 -10757, -7380.5..."
2,3,A St,major,A31,"LINESTRING (-7380.5 -10632.5, -7263.6999999999..."
3,4,A St,major,A31,"LINESTRING (-7263.699999999953 -10497.5, -7133..."
4,5,A St,major,A31,LINESTRING (-7133.599999999977 -10349.70000000...


### 7.4 Do the streets line up with the buildings?

A numeric check before looking: streets run **between** buildings, so points along the street centerlines should almost never fall inside a building's footprint (Notebook 6, Step 7). If the conversion were off by even 40 ft, many would.

**What this does:** takes a point every 50 ft along every segment inside our tiles, and counts the share that falls inside a building footprint. Then the same with the streets **moved on purpose** by 40 ft east, west, north or south: the true position should give by far the lowest share.

**What to look for:** about **0.5%** at no shift (bridges, arcades, buildings over a street), and about **12–13%** for every 40 ft shift. So the streets sit where they should, to well within 40 ft.

In [19]:
outlines = []
for tile in tile_names:
    for model_id, corner_lists in footprints.load_footprints(tile).items():   # from the cache (Notebook 6)
        outlines.extend(shapely.Polygon(corners).buffer(0) for corners in corner_lists if len(corners) >= 3)
footprint_tree = shapely.STRtree(outlines)            # a search structure: fast "which footprint is this point in?"

street_points = []
for line in segments["line"]:
    steps = max(1, int(line.length // 50))            # a point every ~50 ft
    street_points.extend((line.interpolate(i * line.length / steps).coords[0]) for i in range(steps))
street_points = np.array(street_points)
tile_boxes = np.array([loading.tile_grid_bounds(tile) for tile in tile_names]) - np.repeat(loading.GLOBAL_ORIGIN[:2], 2)
in_tiles = ((street_points[:, 0] > tile_boxes[:, 0].min()) & (street_points[:, 0] < tile_boxes[:, 1].max())
            & (street_points[:, 1] > tile_boxes[:, 2].min()) & (street_points[:, 1] < tile_boxes[:, 3].max()))
street_points = street_points[in_tiles]
print(f"{len(outlines):,} footprints, {len(street_points):,} street points")

12,108 footprints, 28,818 street points


In [20]:
def share_inside_buildings(points, shift):
    moved = shapely.points(points + shift)
    point_numbers, footprint_numbers = footprint_tree.query(moved, predicate="within")   # every (point, footprint) hit
    return len(np.unique(point_numbers)) / len(points)


for shift in [(0, 0), (40, 0), (-40, 0), (0, 40), (0, -40)]:
    print(f"streets moved by {str(shift):10} ft: {share_inside_buildings(street_points, np.array(shift)):5.1%} "
          f"of street points inside a building")

streets moved by (0, 0)     ft:  0.5% of street points inside a building
streets moved by (40, 0)    ft: 12.7% of street points inside a building
streets moved by (-40, 0)   ft: 12.1% of street points inside a building
streets moved by (0, 40)    ft: 13.0% of street points inside a building
streets moved by (0, -40)   ft: 13.3% of street points inside a building


### 7.5 Join segments into streets, and place the labels

One label per block would be far too many. So:
1. **Join:** `streets.join_streets` merges segments with the **same name that touch** into longer lines (shapely's `line_merge`). Two different streets with the same name in different neighborhoods (there are several "A St"-like repeats, and both sides of a divided road) stay separate pieces. Each piece keeps the most important road class of its segments.
2. **Place:** `streets.label_positions` puts about **one label per 1,000 ft** of each piece, evenly spread (a 2,500 ft street gets 2, at ¼ and ¾); a piece shorter than 1,000 ft gets one in its middle; pieces under 150 ft (stubs) get none.
3. **Height:** each label sits **15 ft above the terrain** at its point (a ray straight down onto the terrain, as for the building ground in Notebook 7), so it floats just over the street, also on Beacon Hill.

```
   segments:   ──┬──┬──┬──┬──┬──┬──      (one per block)
   joined:     ─────────────────────     one line, 2,500 ft
   labels:          ▲           ▲         at 1/4 and 3/4
                Federal St  Federal St
```

The labels always **face the camera** (pyvista point labels), so they read from any angle. A label is **hidden when a building stands in front of it** (`always_visible=False`): that alone removes a lot of clutter.

**What to look for:** about 2,050 pieces of 1,400 names; the longest are the Mass Pike and I-93; about 2,000 label positions (~800 of them highway or major). Under a second.

In [21]:
start = time.perf_counter()
named_streets = streets.join_streets(segments)
street_labels = streets.label_positions(tiles, named_streets, spacing_ft=1000, min_length_ft=150)
print(f"{len(named_streets):,} street pieces of {named_streets['name'].nunique():,} names -> "
      f"{len(street_labels):,} labels ({time.perf_counter() - start:.1f} s)")
print(street_labels["road_class"].value_counts().to_dict())
named_streets.drop(columns="line").head(8)

2,046 street pieces of 1,411 names -> 2,000 labels (0.5 s)
{'local': 987, 'major': 728, 'other': 208, 'highway': 77}


,name,road_class,length_ft
0,Massachusetts Tpke W,highway,22834.743294
1,Interstate 93 S,highway,20682.431793
2,Interstate 93 N,highway,20343.257716
3,Massachusetts Tpke E,highway,20224.177266
4,Beacon St,major,10573.724228
5,James J Storrow Memorial Dr,major,9465.705541
6,Shawmut Ave,major,9156.566181
7,James J Storrow Memorial Dr,major,8886.681084


### 7.6 Street names over the aerial photo

**What this does:** a new map view (6.6) looking down on downtown, with the major street names (the `DISPLAY` defaults: highway and major roads, pieces of at least 300 ft).

**What to look for (the plan's "done when"):** names on the right streets of the aerial photo. Easy checks:
- *Tremont St* and *Boylston St* along the edges of the Common;
- *Atlantic Ave* along the waterfront, *Congress St* and *Summer St* crossing Fort Point Channel;
- *Massachusetts Tpke* and *Interstate 93* on the highways.

Zoom in: labels hidden behind buildings appear as you move. Clicks and the annotation form still work in this view.

In [38]:
downtown_view = viewing.building_view(tiles, locations, TEST_BUILDINGS["100 Federal"], from_angle=20, down_angle=55)
show_map(downtown_view)

Widget(value='<iframe src="http://localhost:50950/index.html?ui=P_0x175085d90_0&reconnect=auto" class="pyvista…

HTML(value='301 - 319 COLUMBUS AV<br>A minor wall (19 ft wide, under 20 ft): no window grid.')

HTML(value='')

---
## Step 8: less clutter, and show/hide controls

### 8.1 Controls

Phase 3 already used ipywidgets controls (Notebook 6), so the toggles are checkboxes and sliders under the view, connected to `DISPLAY` (6.4):

| Control | Setting |
|---|---|
| Annotations | `annotations` |
| Street names | `street_names` |
| Only major streets | `street_classes`: highway + major, or all |
| Min. street length | `min_street_length_ft` (0–3,000 ft): hides short streets |
| Tallest buildings | `tallest_buildings`: name the 0–30 tallest |

Changing any of them updates `DISPLAY` and calls `refresh_overlays()`. You can also set `DISPLAY[...]` in code and call `refresh_overlays()`.

**What to look for:** the controls under this cell. With the defaults, about 730 street labels exist (many hidden behind buildings at any moment). Untick *Only major streets* and set *Min. street length* to 0: all ~2,000. Move *Min. street length* to 2,000: only long streets. *Tallest buildings* 10: names over the skyline, as the catalog spells them (*John Hancock Tower: Restaurant/Retail/Parking*, *Prudential Redevelopment (Parcel 4 and 6)*, *1 Dalton Street*, *Millennium Tower*, *201 WASHINGTON ST*, …).

In [23]:
# The controls (boilerplate: what each one looks like)
annotations_box = ipywidgets.Checkbox(value=DISPLAY["annotations"], description="Annotations")
streets_box = ipywidgets.Checkbox(value=DISPLAY["street_names"], description="Street names")
major_box = ipywidgets.Checkbox(value=DISPLAY["street_classes"] is not None, description="Only major streets")
length_slider = ipywidgets.IntSlider(value=DISPLAY["min_street_length_ft"], min=0, max=3000, step=100,
                                     description="Min. length (ft)", style={"description_width": "initial"})
tallest_slider = ipywidgets.IntSlider(value=DISPLAY["tallest_buildings"], min=0, max=30,
                                      description="Tallest buildings", style={"description_width": "initial"})

In [24]:
def controls_changed(change):
    DISPLAY["annotations"] = annotations_box.value
    DISPLAY["street_names"] = streets_box.value
    DISPLAY["street_classes"] = ["highway", "major"] if major_box.value else None
    DISPLAY["min_street_length_ft"] = length_slider.value
    DISPLAY["tallest_buildings"] = tallest_slider.value
    refresh_overlays()                                # redraw with the new settings


for control in [annotations_box, streets_box, major_box, length_slider, tallest_slider]:
    control.observe(controls_changed, names="value")  # call controls_changed whenever its value changes
display(ipywidgets.VBox([ipywidgets.HBox([annotations_box, streets_box, major_box]),
                         ipywidgets.HBox([length_slider, tallest_slider])]))

### 8.2 How do labels affect smoothness?

**What this does:** with the view from 7.6 still open, draws it 10 times for each setting and reports the average **frame time** (how long one drawing takes), with all 12 tiles:
- nothing extra; + annotations; + major street names; + all street names; + the 10 tallest buildings' names.

Then it puts your settings back.

**What to look for:** the jump when street names are switched on. Labels that hide behind buildings (`always_visible=False`) must check, every frame, whether something is in front of them, so **many labels cost time**. Under ~50 ms per frame feels smooth; if *all street names* is much slower, keep *Only major streets* on (the default).

*I couldn't measure this myself (no display in my sandbox): please tell me your numbers.*

In [25]:
def average_frame_ms(repeats=10):
    start = time.perf_counter()
    for i in range(repeats):
        map_plotter.render()                          # draw the view once
    return 1000 * (time.perf_counter() - start) / repeats


your_settings = dict(DISPLAY)                         # a copy, to put back afterwards
steps = [("nothing extra", {"annotations": False, "street_names": False, "tallest_buildings": 0}),
         ("+ annotations", {"annotations": True}),
         ("+ major street names", {"street_names": True, "street_classes": ["highway", "major"], "min_street_length_ft": 300}),
         ("+ all street names", {"street_classes": None, "min_street_length_ft": 0}),
         ("+ 10 tallest names", {"tallest_buildings": 10})]
rows = []
for description, changes in steps:
    DISPLAY.update(changes)
    refresh_overlays()
    rows.append({"Showing": description, "Street labels": len(shown_street_labels), "ms per frame": round(average_frame_ms())})
DISPLAY.update(your_settings)
refresh_overlays()
pd.DataFrame(rows)

,Showing,Street labels,ms per frame
0,nothing extra,0,3
1,+ annotations,0,2
2,+ major street names,732,3
3,+ all street names,2000,8
4,+ 10 tallest names,2000,10


---
## Step 9: view check and sun exposure

Two questions about an annotated window (the example: *my flat*, Hancock tower, floor ~19, south wall):

| Part | Question | Answer, as |
|---|---|---|
| **9.1 View check** | Looking out of the window, which buildings block the view, and which directions are open? | colored rays in the view + a table |
| **9.2 Sun exposure** | On a given day, how many hours of direct sun does the window get, and when? | a timeline bar + hours; the whole wall colored by hours |

**Both are ray tests.** A ray from the window either meets a building or it doesn't:

```
   view check:   window ●───────────────■ building       (a fan of level rays, every 2°)
   sun:          window ●─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ☀           (one ray toward the sun, every 10 minutes)
                         └── meets a building? → shade
```

New code: **`citymap/rays.py`** (the ray tests and the view check) and **`citymap/sun.py`** (where the sun is, and sun hours). Both reuse what exists: `windows.locate` gives the window point (2 ft in front of the wall) and the wall's bearing; the ray search structure is the one from Notebook 7b, 2.3 (`vtkModifiedBSPTree`), now one per tile; `building_index` turns a hit into a building; bearings use **true north** (the 0.34° correction from Notebook 7b, 2.5).

### Where the data ends

Only our 12 tiles have buildings. A ray that leaves them without hitting anything is **not** called "open":
- view check: it's **unknown** (gray);
- sun: it's **"sun?"** if it leaves our data lower than 300 ft (`sun.SUN_SETTINGS["unknown_below_ft"]`), because a building just outside could block it.

That way missing neighbors show up in the results instead of silently counting as sun or open view. **Do we need more tiles?** See 9.3 at the end.

### Limits (all approximate)

- No trees, and the terrain is from 2011.
- Window positions are estimates (floor heights from Notebook 7).
- The sun formula leaves out refraction (near the horizon the sun looks ~0.5° higher than it is), so the first and last few minutes of the day may be off.
- "Sun" means the sun is in front of the wall and nothing blocks it, even when it only grazes the wall at a low angle.

**What this does:** builds the ray search structures for the 12 tiles, and the data area.

**What to look for:** about 1 s. Data area x from −22,500 to −2,500 and y from −17,500 to −2,500 (20,000 × 15,000 ft, the 4 × 3 block).

In [26]:
start = time.perf_counter()
locators = rays.build_locators(tiles)                 # one ray search structure per tile (Notebook 7b, 2.3)
area = rays.model_area(tile_names)                    # (x_min, x_max, y_min, y_max): where our data ends
print(f"Ray search structures for {len(locators)} tiles in {time.perf_counter() - start:.1f} s")
print(f"Data area: x {area[0]:,.0f} to {area[1]:,.0f}, y {area[2]:,.0f} to {area[3]:,.0f} "
      f"({area[1] - area[0]:,.0f} x {area[3] - area[2]:,.0f} ft)")

Ray search structures for 12 tiles in 0.8 s
Data area: x -22,500 to -2,500, y -17,500 to -2,500 (20,000 x 15,000 ft)


---
### 9.1 View check

#### 9.1.1 A fan of rays from *my flat*

**What this does:** `rays.view_check` shoots **level** rays from the window (2 ft in front of the wall), **every 2°**, over the half of the compass the wall faces: from the wall's bearing − 88° to + 88° (straight along the wall is left out). That's 89 rays. Each ray goes up to **2 miles** (`rays.RAY_SETTINGS`), and gets one of three outcomes:

| Outcome | Meaning | Drawn |
|---|---|---|
| **blocked** | it meets a building; we keep which one and how far away | red |
| **open** | nothing within 2 miles | green |
| **unknown** | it leaves our 12 tiles before 2 miles, without meeting anything: no data beyond | gray |

The first line shows how far the window is from the edge of our data in each direction.

**What to look for:** the edges about 10,000 ft away to the west and east, 9,600 ft south and 5,400 ft north. 89 rays in a fraction of a second: **33 blocked, 30 open, 26 unknown**. The unknown rays point roughly south: the south edge (~9,600 ft) is closer than 2 miles (10,560 ft), so they leave our data first. Inside our data, those directions are open.

To look up or down instead of level, pass e.g. `elevations=(-10, 0, 10)`.

In [27]:
point = example_spot["point"]
print(f"From the window to the edge of our data: west {point[0] - area[0]:,.0f} ft, east {area[1] - point[0]:,.0f} ft, "
      f"south {point[1] - area[2]:,.0f} ft, north {area[3] - point[1]:,.0f} ft")

start = time.perf_counter()
fan = rays.view_check(locators, tiles, area, example_spot)       # core: one row per ray
fan["name"] = [building_name(model_id) if pd.notna(model_id) else "" for model_id in fan["model_id"]]
print(f"{len(fan)} rays in {time.perf_counter() - start:.2f} s: {fan['outcome'].value_counts().to_dict()}")

From the window to the edge of our data: west 9,975 ft, east 10,025 ft, south 9,568 ft, north 5,432 ft
89 rays in 0.26 s: {'blocked': 33, 'open': 30, 'unknown': 26}


#### 9.1.2 Which buildings block the view?

**What this does:** two tables.
1. **One row per blocking building:** how many rays it stops (2° each, so 12 rays = 24° of the view), from which bearing to which, how close it comes, and its height. Sorted by how much of the view it takes.
2. **Every 10°:** what the window sees in that direction.

**What to look for:**
- one unnamed 380 ft building takes much more of the view than the others: 12 rays (24°, bearings 97°–119°) at only ~240 ft;
- *Liberty Mutual* (east, ~830 ft away) and the *Prudential Redevelopment* tower (southwest, ~1,200 ft, 766 ft tall) take 7 rays each;
- seven far-away buildings stop single rays, from ~2,600 ft (*W Boston Hotel*) to ~9,600 ft (*Brigham and Women's Hospital*);
- in the second table, from 71° (east-northeast) to 241° (west-southwest): blocked to the east, open to the southeast and southwest, unknown straight south (161°–201°, where the edge of our data is ~9,600–10,250 ft away), and the Prudential at 241°.

In [28]:
blocked = fan[fan["outcome"] == "blocked"]
blockers = (blocked.groupby("model_id", sort=False)            # one row per blocking building, in fan order
            .agg(name=("name", "first"), rays=("bearing", "size"), from_bearing=("bearing", "first"),
                 to_bearing=("bearing", "last"), nearest_ft=("distance_ft", "min"))
            .reset_index())
blockers["height_ft"] = [lookup.get_building(catalog, model_id)["Height_Ft"] for model_id in blockers["model_id"]]
blockers.sort_values("rays", ascending=False).round(0).reset_index(drop=True)

,model_id,name,rays,from_bearing,to_bearing,nearest_ft,height_ft
0,BOS-OR18L-1YR0D-7ULH9,(no name),12,97.0,119.0,243.0,380.0
1,BOS-TWTV3-G9KPS-5D59H,Liberty Mutual,7,77.0,89.0,833.0,343.0
2,BOS-KBIA4-2XA04-WNETP,Prudential Redevelopment (Parcel 4 and 6),7,233.0,247.0,1213.0,766.0
3,BOS-DNK4U-VCA5X-L1JBF,Kensington Place - Hinge Block,1,71.0,71.0,3249.0,323.0
4,BOS-TIX06-Q3SHD-SUT3H,45 Stuart Street,1,73.0,73.0,3043.0,349.0
5,BOS-KW3NK-5PL1E-DDV8Z,W Boston Hotel & Residences,1,75.0,75.0,2602.0,325.0
6,BOS-T0VXM-6OGFG-WSJOJ,1 NASSAU ST,1,91.0,91.0,3170.0,267.0
7,BOS-G7Z2S-BMZ3Y-F4N8X,MassPort Parcel F-1 Office Building,1,95.0,95.0,8880.0,269.0
8,BOS-WYTYL-PXUAN-ZLU71,Northeastern University - Columbus Avenue Stud...,1,213.0,213.0,5120.0,272.0
9,BOS-XFBKR-9L3KX-5UV9C,Brigham and Women's Hospital,1,239.0,239.0,9577.0,343.0


In [29]:
every_10 = fan.iloc[::5]                              # rays are 2 degrees apart: every 5th = every 10 degrees
every_10[["bearing", "outcome", "name", "distance_ft"]].round(0).reset_index(drop=True)

,bearing,outcome,name,distance_ft
0,71.0,blocked,Kensington Place - Hinge Block,3249.0
1,81.0,blocked,Liberty Mutual,848.0
2,91.0,blocked,1 NASSAU ST,3170.0
3,101.0,blocked,(no name),295.0
4,111.0,blocked,(no name),251.0
5,121.0,open,,10560.0
6,131.0,open,,10560.0
7,141.0,open,,10560.0
8,151.0,open,,10560.0
9,161.0,unknown,,10116.0


#### 9.1.3 In the view

**What this does:** a map view (6.6) from high up behind the tower, looking out over the fan: rays **red** where blocked (they end on the building), **green** where open (2 miles long), **gray** where they leave our data. The blocking buildings are **orange**.

**What to look for:** a short red patch close to the tower on the east side (the unnamed neighbor at ~240 ft), more red toward Liberty Mutual and the Prudential tower, green rays to the southeast and southwest, and gray rays straight south ending at the south edge of the tiles. Clicks still work: click an orange building to see its name.

In [30]:
def draw_view_check(plotter):
    rays.add_view_rays(plotter, example_spot["point"], fan)                     # red / green / gray rays
    viewing.color_buildings(tiles, blockers["model_id"], STYLE_BLOCKER_COLOR)  # the blocking buildings


STYLE_BLOCKER_COLOR = "#ff8c00"                       # orange, as the groups in Notebook 6
out = np.array([example_spot["frame"]["direction"][0], example_spot["frame"]["direction"][1], 0.0])   # the wall's way
fan_view = (example_spot["point"] - 2500 * out + np.array([0.0, 0.0, 5000.0]),     # behind the tower, high up
            example_spot["point"] + 3000 * out,                                     # looking out over the fan
            np.array([0.0, 0.0, 1.0]))
show_map(fan_view, extra=draw_view_check)

Widget(value='<iframe src="http://localhost:50950/index.html?ui=P_0x174d407d0_0&reconnect=auto" class="pyvista…

HTML(value='Click a wall to describe it, or a marker to read its note.')

HTML(value='')

---
### 9.2 Sun exposure

#### 9.2.1 Where is the sun? Check the formula first

**How the sun's position is computed** (`sun.sun_position`): NOAA's solar position equations (from the "General Solar Position" spreadsheet of the US National Oceanic and Atmospheric Administration), written out in about 30 lines of numpy. **No new library.** In short:

1. the date and time (converted to UTC) give the number of days since 1 January 2000;
2. from that, where the Earth is on its orbit, and so the sun's **declination**: how far north or south of the equator it stands (−23.44° on Dec 21, 0° at the equinoxes, +23.44° on Jun 21);
3. the **equation of time**: how many minutes the sun runs ahead of or behind the clock (up to ±16 minutes during the year);
4. with Boston's longitude, the **hour angle**: how far the sun is from its highest point;
5. spherical trigonometry turns declination, hour angle and latitude into **elevation** and **azimuth**.

Times are Boston time with daylight saving time, from Python's own `zoneinfo` (no library either).

**What this does:** checks the formula on three days, before trusting it:
- **sunrise and sunset**, by the official definition (the top of the sun's disk on the horizon: elevation −0.833°), to compare with any almanac;
- the **highest elevation**, against simple geometry: at solar noon the sun stands 90° − latitude + declination above the horizon;
- the sun at solar noon must be **due south** (azimuth 180°).

**What to look for:**
- sunrise/sunset about **7:11/16:14** on Dec 21, **6:48/18:56** on Mar 20 and **5:08/20:24** on Jun 21. Published times (e.g. timeanddate.com for Boston) are about 7:10/16:15 and 5:07/20:25: within a minute or two;
- highest elevation equal to the expected one (24.2°, 47.6°, 71.1°) to within about 0.05°;
- azimuth 180° ± 0.1 at solar noon, which comes at about 11:42 in winter and 12:46 in summer (daylight saving time).

In [31]:
hancock_row = lookup.get_building(catalog, TEST_BUILDINGS["Hancock"])
LATITUDE, LONGITUDE = float(hancock_row["Centr_Lat"]), float(hancock_row["Centr_Lon"])   # the catalog has them
DAYS = {"winter solstice": date(2026, 12, 21), "equinox": date(2026, 3, 20), "summer solstice": date(2026, 6, 21)}
EARTH_TILT = 23.44                                    # degrees: how far the sun moves north and south of the equator

rows = []
for (label, day), declination in zip(DAYS.items(), [-EARTH_TILT, 0.0, EARTH_TILT]):
    sunrise, sunset = sun.sunrise_sunset(day, LATITUDE, LONGITUDE)
    minute_steps = sun.sun_steps(day, LATITUDE, LONGITUDE, step_minutes=1)
    highest = minute_steps.loc[minute_steps["elevation"].idxmax()]          # solar noon
    rows.append({"day": f"{label} ({day:%b %d})", "sunrise": f"{sunrise:%H:%M %Z}", "sunset": f"{sunset:%H:%M}",
                 "solar noon": f"{highest['time']:%H:%M}", "sun there (azimuth)": round(highest["azimuth"], 1),
                 "highest (elevation)": round(highest["elevation"], 2),
                 "expected highest": round(90 - LATITUDE + declination, 2)})
print(f"Latitude {LATITUDE:.4f}, longitude {LONGITUDE:.4f}")
pd.DataFrame(rows)

Latitude 42.3492, longitude -71.0752


,day,sunrise,sunset,solar noon,sun there (azimuth),highest (elevation),expected highest
0,winter solstice (Dec 21),07:11 EST,16:14,11:42,179.9,24.21,24.21
1,equinox (Mar 20),06:48 EDT,18:56,12:52,180.1,47.69,47.65
2,summer solstice (Jun 21),05:08 EDT,20:24,12:46,179.9,71.09,71.09


### 9.2.2 Sun on *my flat*: three days of the year

**What this does:** for each day, the sun's position every 10 minutes (`sun.sun_steps`), then for each step (`sun.exposure`):

```
  sun below the horizon?                          → night
  sun on the other side of the wall?              → behind wall   (its direction vs the wall's: angle over 90°)
  ray from the window toward the sun meets a building?  → shade   (the building is remembered)
  ray leaves our data lower than 300 ft?          → sun?          (a building outside could block it)
  otherwise                                       → sun
```

The ray stops where nothing can block it any more: higher than every building (1,000 ft) or at the edge of our data. Each day is shown as a bar, midnight to midnight, one slice per 10 minutes (hover over a slice for its time). The table adds up the hours.

The second cell lists the buildings that shade the window on the winter solstice, and when.

**What to look for:**

| Day | sun | shade | behind wall | night |
|---|---|---|---|---|
| winter solstice | 8.3 h | 0.5 h | 0 | 15.2 h |
| equinox | 8.7 h | 1.3 h | 2.0 h | 12.0 h |
| summer solstice | 8.8 h | 0 | 6.2 h | 9.0 h |

- **winter:** sun from sunrise to about 15:40, then the *Prudential Redevelopment* tower (southwest, 766 ft) shades the window until sunset (15:40–16:00 in the second table). The wall faces 159° (south-southeast) and the winter sun stays between 123° and 236°, always in front of the wall: no "behind wall".
- **summer:** sun through the middle of the day, and about 6 h "behind wall": early mornings and evenings the sun is in the northeast and northwest, behind a south-facing wall.
- no "sun?" on any day: toward the south our data reaches ~9,600 ft, far enough (9.3).

In [32]:
timelines = {}
rows = []
for label, day in DAYS.items():
    steps_of_day = sun.sun_steps(day, LATITUDE, LONGITUDE)                            # every 10 minutes
    timelines[label] = sun.exposure(locators, tiles, area, example_spot, steps_of_day)   # core: sun or not, per step
    display(ipywidgets.HTML(f"<b>{label}, {day:%b %d}</b>" + sun.timeline_html(timelines[label])))
    rows.append({"day": label, **sun.hours(timelines[label])})
pd.DataFrame(rows).set_index("day")

HTML(value='<b>winter solstice, Dec 21</b><div style="width:720px;font:12px sans-serif"><div style="display:fl…

HTML(value='<b>equinox, Mar 20</b><div style="width:720px;font:12px sans-serif"><div style="display:flex;heigh…

HTML(value='<b>summer solstice, Jun 21</b><div style="width:720px;font:12px sans-serif"><div style="display:fl…

,sun,sun?,shade,behind wall,night
day,,,,,
winter solstice,8.33,0.0,0.50,0.00,15.17
equinox,8.67,0.0,1.33,2.00,12.00
summer solstice,8.83,0.0,0.00,6.17,9.00


In [33]:
winter = timelines["winter solstice"]
shaded = winter[winter["state"] == "shade"]
if len(shaded) == 0:
    print("No shade on the winter solstice.")
else:
    by_building = shaded.groupby("model_id").agg(first=("time", "min"), last=("time", "max"), steps=("time", "size")).reset_index()
    by_building["name"] = by_building["model_id"].map(building_name)
    by_building["first"] = by_building["first"].dt.strftime("%H:%M")
    by_building["last"] = by_building["last"].dt.strftime("%H:%M")
    by_building["hours"] = by_building["steps"] * sun.SUN_SETTINGS["step_minutes"] / 60
    display(by_building[["name", "first", "last", "hours", "model_id"]])

,name,first,last,hours,model_id
0,Prudential Redevelopment (Parcel 4 and 6),15:40,16:00,0.5,BOS-KBIA4-2XA04-WNETP


### 9.2.3 Check: a south wall with nothing in front of it

**What this does:** the check the proposal asked for. Near the top of the same tower (floor 58, south wall, center column) nothing in our data is taller in front of it, so on the winter solstice, when the sun stays in the southern sky all day, the window should get **sun all day long**.

**What to look for:** sun for all 8.8 h of daylight, no shade, no "behind wall". (The 10-minute steps round the daylight a little: 8.8 h, against 9.0 h from sunrise to sunset in 9.2.1.)

In [34]:
top_spot = windows.locate(tiles, locations, floor_data, TEST_BUILDINGS["Hancock"], 58, "S", "center")
print(top_spot["description"])
top_timeline = sun.exposure(locators, tiles, area, top_spot, sun.sun_steps(DAYS["winter solstice"], LATITUDE, LONGITUDE))
top_hours = sun.hours(top_timeline)
daylight = top_hours["sun"] + top_hours["sun?"] + top_hours["shade"] + top_hours["behind wall"]
print(f"Dec 21: sun {top_hours['sun']} h of {daylight} h of daylight; shade {top_hours['shade']} h")
display(ipywidgets.HTML(sun.timeline_html(top_timeline)))

Floor ~58 of 60 (manual count, level estimated), south-facing wall (159°), column 4 of 6 from the left
Dec 21: sun 8.83 h of 8.83 h of daylight; shade 0.0 h


HTML(value='<div style="width:720px;font:12px sans-serif"><div style="display:flex;height:22px;border:1px soli…

### 9.2.4 The whole wall on the winter solstice

**What this does:** `sun.wall_sun_hours` runs the same test for **every cell** of the window grid on *my flat*'s wall (60 floors × 6 columns = 360 cells), with 20-minute steps (twice as fast as 10, still fine for a picture). Only the steps when the sun is up and in front of the wall are tested. Then the wall is drawn with each cell colored by its hours of sun (dark blue = none, yellow = most), with a color bar.

**What to look for:** about 5–7 s. Hours from 1 (the lowest floors, shaded by the neighbors most of the day) up to 9 (all daylight). The average per floor rises steadily from the bottom: 1.4 h on floor 1, 4.9 on floor 7, 7.4 on floor 13, 8.1 on floor 19, 8.7 on floor 25, and 9 from floor 31 up. In the view: a dark band at the bottom that fades to yellow higher up.

In [35]:
start = time.perf_counter()
winter_steps_20 = sun.sun_steps(DAYS["winter solstice"], LATITUDE, LONGITUDE, step_minutes=20)
wall_cells = sun.wall_sun_hours(locators, tiles, area, example_spot, winter_steps_20)       # core
print(f"{len(wall_cells)} cells in {time.perf_counter() - start:.1f} s.  Hours of sun: "
      f"least {wall_cells['sun_hours'].min():.1f}, median {wall_cells['sun_hours'].median():.1f}, "
      f"most {wall_cells['sun_hours'].max():.1f}.  'sun?' in any cell: {wall_cells['unsure_hours'].max():.1f} h")
wall_cells.groupby("floor")["sun_hours"].mean().round(1).iloc[::6].rename("average hours of sun (every 6th floor)")

360 cells in 5.5 s.  Hours of sun: least 1.0, median 9.0, most 9.0.  'sun?' in any cell: 0.0 h


floor
1     1.4
7     4.9
13    7.4
19    8.1
25    8.7
31    9.0
37    9.0
43    9.0
49    9.0
55    9.0
Name: average hours of sun (every 6th floor), dtype: float64

In [36]:
middle_cell = wall_cells.iloc[len(wall_cells) // 2]          # a cell half way up the wall, for the camera
show_map(windows.wall_view(example_spot["frame"], middle_cell, distance=1600),
         extra=lambda plotter: sun.add_wall_hours(plotter, example_spot, wall_cells))

Widget(value='<iframe src="http://localhost:50950/index.html?ui=P_0x1388c4f50_0&reconnect=auto" class="pyvista…

HTML(value='Click a wall to describe it, or a marker to read its note.')

HTML(value='')

---
### 9.3 Do we need more tiles around the 12?

**For most spots: no.** What decides it is how close a window is to the **edge of our data**, and in which direction it looks.

**Sun.** A building outside the tiles can only shade a window if it is tall enough for the distance. With the sun at elevation *e*, a building *H* ft taller than the window shades it from up to *H* / tan(*e*) away:

| Sun elevation | When (Boston) | A building 300 ft taller than the window shades it from up to |
|---|---|---|
| 24° | winter noon | ~670 ft |
| 10° | winter, ~1.5 h after sunrise | ~1,700 ft |
| 5° | the first / last half hour | ~3,400 ft |

So a window **more than ~2,000 ft from the edge** in the sun's direction is only affected in the first and last minutes of the day. Those are exactly the steps marked **"sun?"**. *My flat* is 9,600 ft from the south edge: 0 h of "sun?" on all three days.

**View.** Beyond the edge we know nothing, and the view check marks those rays **unknown** (gray), not open. From *my flat*, the gray rays point south, where the edge is ~9,600 ft away (just under the 2-mile limit). More tiles (row 7) would only turn those far gray rays into green or red.

**When to add tiles:**
- an annotation shows **"sun?" hours** in 9.2;
- or a lot of **gray** in its view check, in directions you care about;
- or it sits within ~2,000 ft of an edge.

Then add only the tiles on that side: column E to the west, J to the east, row 3 to the north, row 7 to the south. It's the same download as the others (Notebook 3), about 100–400 MB each. The checks above tell you whether it changed anything.

---
## Summary of Notebook 8

- **Annotations** (`citymap/annotations.py`): saved as descriptions in `data/annotations/annotations.json` (gitignored); added in code or by clicking a wall; edited and deleted by id; placed again on every run with `windows.locate`. Clicking a marker shows its note.
- **Street names** (`citymap/streets.py`): City of Boston street segments (PDDL), already in State Plane feet; conversion checked against the catalog (7.1) and the footprints (7.4); joined per street, about one label per 1,000 ft, 15 ft above the terrain.
- **Clutter:** major streets only by default, a minimum length, labels hidden behind buildings, checkboxes and sliders for all overlays; frame times measured in 8.2.
- **View check** (`citymap/rays.py`): a fan of rays from a window; each ray is blocked (which building, how far), open (2 miles), or unknown (it leaves our data). Shown as colored rays, with the blocking buildings in orange, plus tables.
- **Sun exposure** (`citymap/sun.py`): NOAA's sun position formula (no new library), checked against sunrise/sunset times and simple geometry; per 10 minutes sun / shade / behind wall / night, with "sun?" where a building outside our data could block it; a timeline bar, hours per day, which buildings shade the window, and the whole wall colored by hours of sun.
- **More tiles:** not needed for spots well inside the block; the "sun?" hours and gray rays show when they would be (9.3).